# DREAM AI — native training in DREAM NOTEBOOK

This is DREAM's native, from-scratch training workflow for the first-party DREAM NOTEBOOK at `https://dream-host-live.onrender.com/notebook`.

- DREAM NOTEBOOK owns the cells, execution UI and workflow.
- A connected Lightning AI Studio GPU worker supplies CUDA compute.
- Training uses DREAM's own transformer initialized from random weights. It does not load Qwen or another foundation model.
- Prepared data and resumable model checkpoints are restored from and backed up to the existing private DREAM Cloudflare R2 vault.
- Training refuses data without the existing provenance, rights, privacy and human-review gates.

The Studio workspace and notebook files persist between Studio restarts. R2 remains the authoritative off-machine backup for prepared data and checkpoints. Free GPU sessions still have provider limits; see the Lightning setup guide in this branch.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
import requests
import torch

DREAM_HOST = os.environ.get("DREAM_HOST_URL", "https://dream-host-live.onrender.com").rstrip("/")
ARTIFACT_TOKEN = os.environ.get("DREAM_NOTEBOOK_ARTIFACT_TOKEN", "").strip()
if not ARTIFACT_TOKEN:
    raise RuntimeError("The connected worker has no DREAM_NOTEBOOK_ARTIFACT_TOKEN. Add that private environment variable to the Lightning Studio and restart the worker.")
if not torch.cuda.is_available():
    raise RuntimeError("DREAM native training requires a connected CUDA GPU. Start the connected Lightning AI Studio GPU worker first.")

print("DREAM compute:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

WORKSPACE = Path(os.environ.get("DREAM_NOTEBOOK_WORKSPACE", str(Path.home() / "dream-ai-workspace"))).expanduser().resolve()
WORKSPACE.mkdir(parents=True, exist_ok=True)
REPO = Path(os.environ.get("DREAM_NOTEBOOK_REPO", str(WORKSPACE / "dream-ai-native"))).expanduser().resolve()
REPO_BRANCH = os.environ.get("DREAM_REPO_BRANCH", "dream-permanent-compute")
DATASET_DIR = Path(os.environ.get("DREAM_NOTEBOOK_DATASETS", str(REPO / "datasets"))).expanduser().resolve()
DATASET_DIR.mkdir(parents=True, exist_ok=True)
if not (REPO / ".git").is_dir():
    if REPO.exists():
        shutil.rmtree(REPO)
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
        "https://github.com/mjnr0477/dream-ai.git", str(REPO)
    ], check=True)
os.chdir(REPO)

PREPARED_DIR = REPO / "data" / "prepared"
CHECKPOINT_DIR = WORKSPACE / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PRETRAIN_CHECKPOINT = CHECKPOINT_DIR / "dream-bpe.pt"
SFT_BASE_CHECKPOINT = CHECKPOINT_DIR / "dream-bpe-sft-base.pt"
SFT_CHECKPOINT = CHECKPOINT_DIR / "dream-bpe-instruct.pt"
CORPUS_ARCHIVE = WORKSPACE / "dream-native-prepared-corpus.zip"

AUTH = {"Authorization": "Bearer " + ARTIFACT_TOKEN}

def r2_request(method, url, **kwargs):
    last_error = None
    for attempt in range(3):
        try:
            response = requests.request(method, url, timeout=(20, 1800), **kwargs)
            if response.status_code >= 500 and attempt < 2:
                time.sleep(4 * (attempt + 1))
                continue
            response.raise_for_status()
            return response
        except requests.RequestException as exc:
            last_error = exc
            if attempt == 2:
                raise
            time.sleep(4 * (attempt + 1))
    raise RuntimeError("DREAM R2 request failed.") from last_error

def r2_list():
    return r2_request("GET", DREAM_HOST + "/api/v1/notebook/artifacts", headers=AUTH).json().get("artifacts", [])

def r2_restore_latest(name, destination):
    matches = [item for item in r2_list() if item.get("name") == name]
    if not matches:
        print("R2: no saved artifact yet:", name)
        return None
    item = max(matches, key=lambda value: value.get("created_at", ""))
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    digest = hashlib.sha256()
    response = r2_request(
        "GET", DREAM_HOST + "/api/v1/notebook/artifacts/" + item["sha256"],
        headers=AUTH, stream=True
    )
    size = 0
    with temporary.open("wb") as output:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                output.write(chunk)
                digest.update(chunk)
                size += len(chunk)
    if digest.hexdigest() != item["sha256"] or size != item["size_bytes"]:
        temporary.unlink(missing_ok=True)
        raise RuntimeError("R2 artifact checksum or size mismatch: " + name)
    temporary.replace(destination)
    print("R2 restore verified:", name, "bytes:", size, "sha256:", item["sha256"])
    return item

def r2_upload(path, name):
    path = Path(path)
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    expected = digest.hexdigest()
    with path.open("rb") as handle:
        result = r2_request(
            "PUT", DREAM_HOST + "/api/v1/notebook/artifacts/" + name,
            data=handle,
            headers={**AUTH, "Content-Type": "application/octet-stream"}
        ).json()["artifact"]
    if result.get("sha256") != expected or result.get("size_bytes") != path.stat().st_size:
        raise RuntimeError("R2 upload verification failed for " + name)
    print("R2 backup verified:", name, "bytes:", result["size_bytes"], "sha256:", expected)
    return result

# Restore the prepared, hash-verified corpus first; the matching model checkpoint follows.
if not (PREPARED_DIR / "manifest.json").is_file():
    corpus_meta = r2_restore_latest("dream-native-prepared-corpus.zip", CORPUS_ARCHIVE)
    if corpus_meta:
        with zipfile.ZipFile(CORPUS_ARCHIVE) as archive:
            if archive.testzip() is not None:
                raise RuntimeError("Saved DREAM corpus archive failed ZIP CRC validation.")
            names = archive.namelist()
            if "SHA256SUMS.txt" not in names:
                raise RuntimeError("Saved DREAM corpus archive has no file checksum manifest.")
            expected_hashes = {}
            for line in archive.read("SHA256SUMS.txt").decode("utf-8").splitlines():
                expected, separator, relative = line.partition("  ")
                if separator and len(expected) == 64:
                    expected_hashes[relative] = expected
            for relative in names:
                if relative == "SHA256SUMS.txt":
                    continue
                target = (REPO / "data" / relative).resolve()
                if not target.is_relative_to((REPO / "data").resolve()):
                    raise RuntimeError("Unsafe path in saved DREAM corpus archive.")
                payload = archive.read(relative)
                if hashlib.sha256(payload).hexdigest() != expected_hashes.get(relative):
                    raise RuntimeError("Corpus file checksum mismatch: " + relative)
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(payload)
        print("Prepared corpus restored from private R2 storage.")

r2_restore_latest("dream-native-pretrain-checkpoint.pt", PRETRAIN_CHECKPOINT)
r2_restore_latest("dream-native-sft-base-checkpoint.pt", SFT_BASE_CHECKPOINT)
r2_restore_latest("dream-native-sft-checkpoint.pt", SFT_CHECKPOINT)
print("DREAM repository:", REPO)
print("Prepared corpus available:", (PREPARED_DIR / "manifest.json").is_file())
print("Native checkpoint available:", PRETRAIN_CHECKPOINT.is_file())

## 1. Review and prepare the training corpus

Place the human-reviewed `approved-records.jsonl` file in the notebook's `datasets/` folder using the notebook dataset tools or the Studio terminal. Every record must already have real rights, privacy and quality review. Do not mark records approved merely to pass the audit.

When a verified prepared corpus already exists in R2, the setup cell restores it and this cell reuses it. Otherwise, the workflow audits and prepares the reviewed file before training.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

prepared_manifest = PREPARED_DIR / "manifest.json"
if prepared_manifest.is_file():
    print("Using the previously audited and R2-backed corpus.")
    manifest = json.loads(prepared_manifest.read_text(encoding="utf-8"))
else:
    candidates = sorted(DATASET_DIR.rglob("approved-records.jsonl"))
    if len(candidates) != 1:
        raise RuntimeError(
              "Place exactly one reviewed approved-records.jsonl file in the notebook datasets/ folder. "
            "Found: " + str(len(candidates))
        )
    source_records = REPO / "data" / "approved-records.jsonl"
    source_records.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(candidates[0], source_records)

    audit_path = REPO / "eval-results" / "native-corpus-audit.json"
    audit_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        sys.executable, "model/audit_corpus.py",
        "--input", str(source_records),
        "--out", str(audit_path)
    ], cwd=REPO, check=True)
    audit = json.loads(audit_path.read_text(encoding="utf-8"))
    print("Corpus audit:", json.dumps(audit, indent=2))
    if audit.get("error_count") or audit.get("possible_pii_findings") or audit.get("unreviewed_records"):
        raise RuntimeError("Corpus failed the privacy, provenance or human-review gate. No training was started.")

    subprocess.run([
        sys.executable, "model/prepare_corpus.py",
        "--input", str(source_records),
        "--output", str(PREPARED_DIR)
    ], cwd=REPO, check=True)
    manifest = json.loads(prepared_manifest.read_text(encoding="utf-8"))

    balance_path = REPO / "eval-results" / "native-language-balance.json"
    subprocess.run([
        sys.executable, "model/audit_language_balance.py",
        "--input", str(source_records),
        "--out", str(balance_path),
        "--min-records", "1000",
        "--max-share", "0.5"
    ], cwd=REPO, check=True)
    balance = json.loads(balance_path.read_text(encoding="utf-8"))
    print("Language balance:", json.dumps(balance, indent=2))

for split in ("train", "validation", "test"):
    split_path = PREPARED_DIR / (split + ".txt")
    if not split_path.is_file() or split_path.stat().st_size < 258:
        raise RuntimeError("Missing or too-small prepared " + split + " split.")
    print(split, "bytes:", split_path.stat().st_size)

print("Corpus manifest SHA-256:", __import__("hashlib").sha256(prepared_manifest.read_bytes()).hexdigest())

## 2. Back up the prepared corpus to DREAM R2

This saves the exact train, validation and held-out test splits, their provenance sidecars, and manifest. Future sessions can therefore resume against the same corpus instead of silently switching datasets.

In [ ]:
from pathlib import Path
import hashlib
import zipfile

if not (PREPARED_DIR / "manifest.json").is_file():
    raise RuntimeError("Prepare and audit the corpus before backing it up.")

files = sorted(path for path in PREPARED_DIR.rglob("*") if path.is_file())
if not files:
    raise RuntimeError("Prepared corpus folder is empty.")

checksums = []
with zipfile.ZipFile(CORPUS_ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        relative = path.relative_to(REPO / "data").as_posix()
        payload = path.read_bytes()
        checksums.append(hashlib.sha256(payload).hexdigest() + "  " + relative)
        archive.write(path, arcname=relative)
    archive.writestr("SHA256SUMS.txt", "\n".join(checksums) + "\n")

with zipfile.ZipFile(CORPUS_ARCHIVE) as archive:
    if archive.testzip() is not None:
        raise RuntimeError("New prepared-corpus ZIP failed CRC validation.")

r2_upload(CORPUS_ARCHIVE, "dream-native-prepared-corpus.zip")
print("Prepared corpus is now backed up in the private R2 vault.")

## 3. Train native DREAM and checkpoint directly to R2

The model starts from random initialization only when no compatible DREAM checkpoint exists. Otherwise, `model/train.py` validates the corpus hash, provenance, tokenizer and training settings before resuming.

The trainer saves atomically every 25 optimizer steps. This cell watches those completed checkpoints and uploads each new one to R2 while training continues. If the GPU worker or Studio session restarts, the next notebook kernel restores the latest successfully uploaded checkpoint.

In [ ]:
from pathlib import Path
import subprocess
import sys
import time

TRAIN_STEPS = 2000
TRAIN_SECONDS = 240
TOKENIZER = "bpe"
VOCAB_SIZE = 1024
MICRO_BATCH = 4
GRAD_ACCUMULATION = 8
BLOCK_SIZE = 256
D_MODEL = 256
LAYERS = 6
HEADS = 8

checkpoint = PRETRAIN_CHECKPOINT
command = [
    sys.executable, "model/train.py",
    "--data", str(PREPARED_DIR / "train.txt"),
    "--validation-data", str(PREPARED_DIR / "validation.txt"),
    "--corpus-manifest", str(PREPARED_DIR / "manifest.json"),
    "--out", str(checkpoint),
    "--tokenizer", TOKENIZER,
    "--vocab-size", str(VOCAB_SIZE),
    "--steps", str(TRAIN_STEPS),
    "--max-seconds", str(TRAIN_SECONDS),
    "--batch-size", str(MICRO_BATCH),
    "--grad-accum-steps", str(GRAD_ACCUMULATION),
    "--block-size", str(BLOCK_SIZE),
    "--d-model", str(D_MODEL),
    "--layers", str(LAYERS),
    "--heads", str(HEADS),
    "--eval-every", "25",
    "--checkpoint-every", "25",
    "--eval-batches", "10"
]
if checkpoint.is_file():
    command += ["--resume", str(checkpoint)]
    print("Resuming the exact native DREAM checkpoint from R2/local recovery.")
else:
    print("Starting native DREAM from random initialization.")

budget_path = REPO / "eval-results" / "native-model-budget.json"
subprocess.run([
    sys.executable, "model/estimate_model.py",
    "--vocab-size", str(VOCAB_SIZE),
    "--block-size", str(BLOCK_SIZE),
    "--d-model", str(D_MODEL),
    "--heads", str(HEADS),
    "--layers", str(LAYERS),
    "--out", str(budget_path)
], cwd=REPO, check=True)
budget = json.loads(budget_path.read_text())
print("Native parameters:", budget["parameter_count"])
print("Memory warning:", budget.get("warning"))

log_path = WORKSPACE / "dream-native-training.log"
last_uploaded_mtime = 0
log_position = 0
upload_failures = []

with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        command, cwd=REPO, stdout=log, stderr=subprocess.STDOUT,
        text=True, env=os.environ.copy()
    )
    while process.poll() is None:
        time.sleep(3)
        log.flush()
        if checkpoint.is_file():
            mtime = checkpoint.stat().st_mtime_ns
            if mtime != last_uploaded_mtime:
                try:
                    r2_upload(checkpoint, "dream-native-pretrain-checkpoint.pt")
                    last_uploaded_mtime = mtime
                except Exception as exc:
                    upload_failures.append(str(exc))
                    print("Checkpoint upload will be retried:", str(exc)[:250])
        with log_path.open("r", encoding="utf-8", errors="replace") as reader:
            reader.seek(log_position)
            new_text = reader.read()
            log_position = reader.tell()
        if new_text:
            print(new_text[-12000:], end="")

    with log_path.open("r", encoding="utf-8", errors="replace") as reader:
        reader.seek(log_position)
        remaining = reader.read()
    if remaining:
        print(remaining[-12000:], end="")

if process.returncode != 0:
    raise subprocess.CalledProcessError(process.returncode, command)

if not checkpoint.is_file():
    raise RuntimeError("Training exited without producing a native DREAM checkpoint.")

# Always upload the final checkpoint after the process stops.
r2_upload(checkpoint, "dream-native-pretrain-checkpoint.pt")
state = __import__("torch").load(checkpoint, map_location="cpu", weights_only=True)
training_state = state.get("training", {})
print("Native DREAM checkpoint saved.")
print("Completed optimizer steps:", training_state.get("completed_steps"))
print("Latest training loss:", training_state.get("last_train_loss"))
print("Latest validation loss:", training_state.get("last_validation_loss"))
if upload_failures:
    print("Some intermediate uploads failed; the final checkpoint upload succeeded.")

## 4. Evaluate the held-out test split

This reports token-prediction metrics and a generated sample. It is a basic training diagnostic, not proof of general assistant capability or factual reliability.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

evaluation_path = REPO / "eval-results" / "dream-native-pretrain.json"
subprocess.run([
    sys.executable, "model/evaluate.py",
    "--checkpoint", str(PRETRAIN_CHECKPOINT),
    "--data", str(PREPARED_DIR / "test.txt"),
    "--batches", "50",
    "--batch-size", "8",
    "--out", str(evaluation_path)
], cwd=REPO, check=True)

evaluation = json.loads(evaluation_path.read_text(encoding="utf-8"))
print("Bits per byte:", evaluation.get("bits_per_byte"))
print("Perplexity per byte:", evaluation.get("perplexity_per_byte"))
print("Sample generation:", evaluation.get("sample_generation"))
r2_upload(evaluation_path, "dream-native-pretrain-evaluation.json")
print("Evaluation report stored in DREAM R2.")

## 5. Optional instruction tuning — teach native DREAM to follow conversations

Pretraining teaches next-token prediction. This optional stage teaches the native model to follow user/assistant conversations. It uses a separate private dataset with `sft-train.jsonl` and `sft-validation.jsonl`.

Every record must have a real source/license, privacy review, quality review and explicit training approval. The audit gate is recreated from the exact file hashes. SFT training is rejected if the source groups overlap the held-out pretraining test split.

This stage starts from a pinned copy of the native DREAM checkpoint, never Qwen. It saves both the SFT base and resumable instruction checkpoint to private R2. If no eligible SFT dataset is attached, it skips instruction tuning without blocking native pretraining.

In [ ]:
from pathlib import Path
import shutil

sft_train_candidates = sorted(DATASET_DIR.rglob("sft-train.jsonl"))
sft_validation_candidates = sorted(DATASET_DIR.rglob("sft-validation.jsonl"))
sft_available = len(sft_train_candidates) == 1 and len(sft_validation_candidates) == 1

if sft_available:
    sft_train_path = REPO / "data" / "sft-train.jsonl"
    sft_validation_path = REPO / "data" / "sft-validation.jsonl"
    sft_train_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(sft_train_candidates[0], sft_train_path)
    shutil.copy2(sft_validation_candidates[0], sft_validation_path)
    print("SFT training bytes:", sft_train_path.stat().st_size)
    print("SFT validation bytes:", sft_validation_path.stat().st_size)
else:
    print("No unique pair of sft-train.jsonl and sft-validation.jsonl was found.")
    print("Native pretraining is complete; instruction tuning is skipped until reviewed SFT data is attached.")


In [ ]:
if sft_available:
    from pathlib import Path
    import hashlib
    import json
    import subprocess
    import sys

    for split_name, split_path in (
        ("train", sft_train_path),
        ("validation", sft_validation_path)
    ):
        report_path = REPO / "eval-results" / f"native-sft-{split_name}-audit.json"
        report_path.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run([
            sys.executable, "model/audit_corpus.py",
            "--input", str(split_path),
            "--out", str(report_path)
        ], cwd=REPO, check=True)
        report = json.loads(report_path.read_text(encoding="utf-8"))
        print(f"SFT {split_name} audit:", json.dumps(report, indent=2))
        if report.get("error_count") or report.get("possible_pii_findings") or report.get("unreviewed_records"):
            raise RuntimeError(f"SFT {split_name} failed DREAM's privacy, provenance or human-review audit.")

    def split_meta(path):
        raw = path.read_bytes()
        return {
            "records": sum(1 for line in raw.splitlines() if line.strip()),
            "sha256": hashlib.sha256(raw).hexdigest()
        }

    sft_manifest = {
        "schema": "dream-sft-splits-v1",
        "dataset_sha256": hashlib.sha256(
            sft_train_path.read_bytes() + sft_validation_path.read_bytes()
        ).hexdigest(),
        "splits": {
            "train": split_meta(sft_train_path),
            "validation": split_meta(sft_validation_path)
        }
    }
    sft_manifest_path = REPO / "eval-results" / "native-sft-split-manifest.json"
    sft_manifest_path.write_text(json.dumps(sft_manifest, indent=2) + "\n", encoding="utf-8")

    subprocess.run([
        sys.executable, "data/verify_dream_ai_sft_audits.py",
        "--manifest", str(sft_manifest_path),
        "--train-audit", str(REPO / "eval-results" / "native-sft-train-audit.json"),
        "--validation-audit", str(REPO / "eval-results" / "native-sft-validation-audit.json"),
        "--output", str(REPO / "eval-results" / "native-sft-audit-gate.json")
    ], cwd=REPO, check=True)
    print("SFT hash-bound audit gate: PASSED")

    if not PRETRAIN_CHECKPOINT.is_file():
        raise RuntimeError("Native pretraining checkpoint is missing; cannot instruction-tune.")
    if not SFT_BASE_CHECKPOINT.is_file():
        shutil.copy2(PRETRAIN_CHECKPOINT, SFT_BASE_CHECKPOINT)
        print("Pinned native DREAM SFT base checkpoint.")
    r2_upload(SFT_BASE_CHECKPOINT, "dream-native-sft-base-checkpoint.pt")
else:
    print("SFT audit and training cells are skipped because no complete reviewed dataset pair is attached.")


In [ ]:
if sft_available:
    import subprocess
    import sys
    import time
    import json
    from pathlib import Path

    command = [
        sys.executable, "model/sft.py",
        "--checkpoint", str(SFT_BASE_CHECKPOINT),
        "--train-data", str(sft_train_path),
        "--validation-data", str(sft_validation_path),
        "--sft-audit-gate", str(REPO / "eval-results" / "native-sft-audit-gate.json"),
        "--pretraining-manifest", str(PREPARED_DIR / "manifest.json"),
        "--pretraining-test-data", str(PREPARED_DIR / "test.txt"),
        "--out", str(SFT_CHECKPOINT),
        "--steps", "1000",
        "--max-seconds", "180",
        "--batch-size", "2",
        "--grad-accum-steps", "4",
        "--eval-every", "25",
        "--checkpoint-every", "25",
        "--eval-batches", "10"
    ]
    if SFT_CHECKPOINT.is_file():
        command += ["--resume", str(SFT_CHECKPOINT)]
        print("Attempting to resume the exact R2-backed native DREAM SFT checkpoint.")

    log_path = WORKSPACE / "dream-native-sft-training.log"
    last_uploaded_mtime = 0
    log_position = 0
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            command, cwd=REPO, stdout=log, stderr=subprocess.STDOUT,
            text=True, env=os.environ.copy()
        )
        while process.poll() is None:
            time.sleep(3)
            log.flush()
            if SFT_CHECKPOINT.is_file():
                mtime = SFT_CHECKPOINT.stat().st_mtime_ns
                if mtime != last_uploaded_mtime:
                    r2_upload(SFT_CHECKPOINT, "dream-native-sft-checkpoint.pt")
                    last_uploaded_mtime = mtime
            with log_path.open("r", encoding="utf-8", errors="replace") as reader:
                reader.seek(log_position)
                new_text = reader.read()
                log_position = reader.tell()
            if new_text:
                print(new_text[-10000:], end="")

        with log_path.open("r", encoding="utf-8", errors="replace") as reader:
            reader.seek(log_position)
            remaining = reader.read()
        if remaining:
            print(remaining[-10000:], end="")

    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, command)
    if not SFT_CHECKPOINT.is_file():
        raise RuntimeError("SFT exited without saving an instruction checkpoint.")
    r2_upload(SFT_CHECKPOINT, "dream-native-sft-checkpoint.pt")

    sft_state = torch.load(SFT_CHECKPOINT, map_location="cpu", weights_only=True)
    sft_training = sft_state.get("sft_training", {})
    print("Native DREAM instruction checkpoint saved.")
    print("Completed SFT optimizer steps:", sft_training.get("completed_steps"))
    print("Latest SFT validation loss:", sft_training.get("last_validation_loss"))
else:
    print("SFT skipped; no reviewed instruction dataset was supplied.")


In [ ]:
if sft_available and SFT_CHECKPOINT.is_file():
    import json
    import subprocess
    import sys

    sft_eval_path = REPO / "eval-results" / "dream-native-instruct.json"
    subprocess.run([
        sys.executable, "model/evaluate.py",
        "--checkpoint", str(SFT_CHECKPOINT),
        "--data", str(PREPARED_DIR / "test.txt"),
        "--batches", "50",
        "--batch-size", "8",
        "--out", str(sft_eval_path)
    ], cwd=REPO, check=True)
    sft_eval = json.loads(sft_eval_path.read_text(encoding="utf-8"))
    print("Native DREAM instruction model bits per byte:", sft_eval.get("bits_per_byte"))
    print("Native DREAM instruction model sample:", sft_eval.get("sample_generation"))
    r2_upload(sft_eval_path, "dream-native-instruction-evaluation.json")
else:
    print("No native DREAM instruction checkpoint is available to evaluate.")

## Completion

Native pretraining is successful only when the trainer reports a completed checkpoint, the checkpoint is uploaded and verified in R2, and the held-out evaluation completes.

The Qwen adapter remains a separate fallback. This workflow does not replace the live chat model automatically. A separate reviewed instruction-tuning and release evaluation is required before native DREAM can serve users.